In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from IPython.display import display, HTML

In [2]:
train = pd.read_csv("DataTraining.csv").drop(columns="date")
test = pd.read_csv("DataTest.csv").drop(columns="date")

In [3]:
!pip install human-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.5/116.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 99.1 MB/s eta 0:00:00
  Attempting uninstall: bokeh
    Found existing installation: bokeh 3.7.3
    Uninstalling bokeh-3.7.3:
      Successfully uninstalled bokeh-3.7.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
panel 1.8.7 requires bokeh<3.9.0,>=3.7.0, but you have bokeh 2.4.3 which is incompatible.
holoviews 1.22.1 requires bokeh>=3.1, but you have bokeh 2.4.3 which is incompatible.


In [4]:
target = "Occupancy"

train_x = train.drop(columns=[target])
train_y = train[target]

val_x = test.drop(columns=[target])
val_y = test[target]

In [5]:
train.head(10)

,Unnamed: 0,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
0,1,23.180,27.2720,426.0,721.250000,0.004793,1
1,2,23.150,27.2675,429.5,714.000000,0.004783,1
2,3,23.150,27.2450,426.0,713.500000,0.004779,1
3,4,23.150,27.2000,426.0,708.250000,0.004772,1
4,5,23.100,27.2000,426.0,704.500000,0.004757,1
5,6,23.100,27.2000,419.0,701.000000,0.004757,1
6,7,23.100,27.2000,419.0,701.666667,0.004757,1
7,8,23.100,27.2000,419.0,699.000000,0.004757,1
8,9,23.100,27.2000,419.0,689.333333,0.004757,1
9,10,23.075,27.1750,419.0,688.000000,0.004745,1


In [ ]:
from IPython.display import display, HTML

display(HTML("""
<h2 style="color:#1f4e79; border-left:6px solid #1f4e79; padding-left:10px;">
    Entraînement et évaluation du modèle Black-Box (Random Forest)
</h2>
"""))

In [6]:
forest_model = RandomForestClassifier(random_state=1)
forest_model.fit(train_x, train_y)
machine_preds = forest_model.predict(val_x)
print(classification_report(val_y, machine_preds))

              precision    recall  f1-score   support

           0       1.00      0.97      0.98      7703
           1       0.91      0.98      0.95      2049

    accuracy                           0.98      9752
   macro avg       0.95      0.98      0.97      9752
weighted avg       0.98      0.98      0.98      9752



***Rule-Based Model***

In [7]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:

display(HTML(f"""
<h2 style="color:#1f4e79; border-left:6px solid #1f4e79; padding-left:10px;">
    Analyse de la feature : Light par rapport à l'occupation
</h2>
<p style="color:#555; font-size:14px;">
    Visualisation de la distribution de la luminosité dans la pièce selon qu'elle soit occupée ou non.
</p>
"""))


In [8]:
import plotly.express as px
import plotly.graph_objects as go

feature = "Light"
px.box(data_frame=train, x=target, y=feature)

In [ ]:
display(HTML(f"""
<h2 style="color:#1f4e79; border-left:6px solid #1f4e79; padding-left:10px;">
    Création et évaluation d'un classifieur basé sur une règle simple (FunctionClassifier)
</h2>
<p style="color:#555; font-size:14px;">
    Nous définissons une règle binaire sur la feature <strong>{feature}</strong> avec un seuil de {threshold},
    et évaluons sa performance sur l'ensemble de validation.
</p>
"""))

In [9]:
import numpy as np
from hulearn.classification import FunctionClassifier

def create_rule(data: pd.DataFrame, col: str, threshold: float):
    return np.array(data[col] > threshold).astype(int)

threshold = 100
mod = FunctionClassifier(create_rule, col=feature, threshold=threshold)
mod.fit(train_x, train_y)
preds = mod.predict(val_x)
print(classification_report(val_y, preds))

              precision    recall  f1-score   support

           0       1.00      0.92      0.96      7703
           1       0.77      1.00      0.87      2049

    accuracy                           0.94      9752
   macro avg       0.88      0.96      0.91      9752
weighted avg       0.95      0.94      0.94      9752



In [ ]:
display(HTML(f"""
<h2 style="color:#1f4e79; border-left:6px solid #1f4e79; padding-left:10px;">
    Visualisation de la feature Light avec seuil pour Rule-Based Learner
</h2>
<p style="color:#555; font-size:14px;">
    Boxplot de la distribution de <strong>Light</strong> selon l'occupation de la pièce,
    avec la ligne rouge indiquant le seuil <strong>100</strong> utilisé pour la règle IF–THEN.
    Cela permet de voir visuellement comment la règle sépare les classes.
</p>
"""))

In [10]:
def plot_threshold(train_df: pd.DataFrame, feature: str, target: str, threshold: float):
    fig = px.box(data_frame=train_df, x=target, y=feature)

    # add a second axis that overlays the existing one
    fig.layout.xaxis2 = go.layout.XAxis(
        overlaying="x", range=[0, 2], showticklabels=False
    )
    fig.add_scatter(
        x=[0, 2],
        y=[threshold, threshold],
        mode="lines",
        xaxis="x2",
        showlegend=False,
        line=dict(dash="dash", color="firebrick", width=2),
    )

    fig.show()

In [11]:
plot_threshold(train, feature, target, threshold)

In [12]:
!pip uninstall numpy -y
!pip install numpy==1.26.4

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 110.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
panel 1.8.7 requi

In [13]:

from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(mod, cv=2, param_grid={"threshold": np.linspace(250, 750, 1000)})
grid.fit(train_x, train_y)

GridSearchCV(cv=2,
             estimator=FunctionClassifier(col='Light',
                                          func=<function create_rule at 0x79600b288ea0>,
                                          threshold=100),
             param_grid={'threshold': array([250.        , 250.5005005 , 251.001001  , 251.5015015 ,
       252.002002  , 252.5025025 , 253.003003  , 253.5035035 ,
       254.004004  , 254.5045045 , 255.00500501, 255.50550551,
       256.00600601, 256.50650651, 257.00700701, 257.50750751,
       258.00800801, 25...
       734.48448448, 734.98498498, 735.48548549, 735.98598599,
       736.48648649, 736.98698699, 737.48748749, 737.98798799,
       738.48848849, 738.98898899, 739.48948949, 739.98998999,
       740.49049049, 740.99099099, 741.49149149, 741.99199199,
       742.49249249, 742.99299299, 743.49349349, 743.99399399,
       744.49449449, 744.99499499, 745.4954955 , 745.995996  ,
       746.4964965 , 746.996997  , 747.4974975 , 747.997998  ,
       748.4984985 , 748.998999  , 749.4994995 , 750.        ])})

In [14]:
best_threshold = grid.best_params_["threshold"]
best_threshold

np.float64(364.61461461461465)

In [15]:
human_preds = grid.predict(val_x)
print(classification_report(val_y, human_preds))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      7703
           1       0.97      0.99      0.98      2049

    accuracy                           0.99      9752
   macro avg       0.99      0.99      0.99      9752
weighted avg       0.99      0.99      0.99      9752



In [16]:
plot_threshold(train, feature, target, best_threshold)

In [ ]:
display(HTML(f"""
<h2 style="color:#1f4e79; border-left:6px solid #1f4e79; padding-left:10px;">
    Combine Two Models
</h2>
"""))

In [17]:
def highlight_cell(row):
    return [
        "background-color: red; color: white"
        if cell == 0
        else "background-color: green; color: white"
        for cell in row
    ]

In [18]:
comparisons = pd.DataFrame(
    {
        "RandomForestClassifier": machine_preds,
        "Rule-based Model": human_preds,
        #         "Real Label": val_y.values,
    }
)

difference = comparisons[
    comparisons["RandomForestClassifier"] != comparisons["Rule-based Model"]
]

difference.head(10)

,RandomForestClassifier,Rule-based Model
1063,1,0
1135,0,1
1136,0,1
1144,0,1
1145,0,1
1146,0,1
1147,0,1
1148,0,1
1149,0,1
1150,0,1


In [19]:
reduce_false_negatives = difference.assign(final_prediction=1)

reduce_false_negatives.style.apply(highlight_cell)

,RandomForestClassifier,Rule-based Model,final_prediction
1063,1,0,1
1135,0,1,1
1136,0,1,1
1144,0,1,1
1145,0,1,1
1146,0,1,1
1147,0,1,1
1148,0,1,1
1149,0,1,1
1150,0,1,1


In [20]:
reduce_false_positives = difference.assign(final_prediction=0)

reduce_false_positives.style.apply(highlight_cell)

,RandomForestClassifier,Rule-based Model,final_prediction
1063,1,0,0
1135,0,1,0
1136,0,1,0
1144,0,1,0
1145,0,1,0
1146,0,1,0
1147,0,1,0
1148,0,1,0
1149,0,1,0
1150,0,1,0


In [ ]:
display(HTML("""
<p style="font-size:15px; line-height:1.6;">
Le principe de la combinaison de deux modèles repose sur l'exploitation des forces complémentaires de chaque approche afin d'améliorer la fiabilité des décisions.
Le modèle de machine learning (Random Forest) permet de capturer des motifs complexes dans les données, tandis que le modèle basé sur des règles fournit une logique interprétable et conservatrice.
Lorsque les deux modèles sont d'accord, la prédiction est considérée comme fiable.
En cas de désaccord, une stratégie prioritaire est appliquée pour réduire les erreurs critiques, en particulier les faux négatifs.
Dans le contexte de la détection de l’occupation des pièces, ne pas détecter une présence réelle est plus problématique que de signaler une présence inexistante.
Ainsi, lorsque les deux modèles divergent, la décision finale est biaisée vers la prédiction d’occupation.
Cette stratégie de combinaison améliore la robustesse et la sécurité tout en conservant l'interprétabilité.
</p>
"""))
